# Experiment: LDA Solvers in Batch vs Streaming Mode (OpenML MNIST)

Objective:
- Compare `LinearDiscriminantAnalysis` solvers (`svd`, `lsqr`, `eigen`) in full-batch (`fit`) and streaming (`partial_fit`) training.
- Measure holdout accuracy, uninstrumented training time, and memory profile over 5-fold stratified cross-validation.
- Use a much larger real dataset from OpenML: `mnist_784` (70,000 samples, 784 features, 10 classes).


In [13]:
from __future__ import annotations

import tracemalloc
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np

from sklearn.datasets import fetch_openml
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit


## Dataset and benchmark configuration

This setup keeps the training protocol fixed while changing only the LDA solver and training mode.


In [21]:
SEED = 42
N_SPLITS = 5
STREAM_BATCH_SIZES = (512, 2048, 8192)
BATCH_FIT_LABEL = 'full-fit'
MEASURE_MEMORY = True

OPENML_DATASET = 'mnist_784'
OPENML_VERSION = 1
# Set to None to run all 70,000 samples.
MAX_SAMPLES = None

SOLVERS = ('svd', 'lsqr', 'eigen')
MODES = ('batch', 'streaming')
# Mild shrinkage improves numerical stability for covariance-based solvers.
SOLVER_PARAMS = {
    'svd': {},
    'lsqr': {'shrinkage': 1e-3},
    'eigen': {'shrinkage': 1e-3},
}


def load_openml_dataset(name: str, version: int):
    # `parser` is available in newer sklearn; fallback keeps compatibility.
    try:
        return fetch_openml(name=name, version=version, as_frame=False, parser="auto")
    except TypeError:
        return fetch_openml(name=name, version=version, as_frame=False)


mnist = load_openml_dataset(OPENML_DATASET, OPENML_VERSION)
X = np.asarray(mnist.data, dtype=np.float32)
y = np.asarray(mnist.target).astype(np.int64, copy=False)
source_samples = X.shape[0]

if MAX_SAMPLES is not None and MAX_SAMPLES < source_samples:
    splitter = StratifiedShuffleSplit(n_splits=1, train_size=MAX_SAMPLES, random_state=SEED)
    subset_idx, _ = next(splitter.split(X, y))
    X = X[subset_idx]
    y = y[subset_idx]

CLASSES = np.unique(y)

print(f"Dataset: OpenML/{OPENML_DATASET} (v{OPENML_VERSION})")
print(f"Samples={X.shape[0]} (from {source_samples}), Features={X.shape[1]}, Classes={len(CLASSES)}")
print(f"Solvers={SOLVERS}, Modes={MODES}, Folds={N_SPLITS}")
print(f"Streaming batch-size sweep={STREAM_BATCH_SIZES}")
print(f"Memory profiling enabled={MEASURE_MEMORY}")


Dataset: OpenML/mnist_784 (v1)
Samples=70000 (from 70000), Features=784, Classes=10
Solvers=('svd', 'lsqr', 'eigen'), Modes=('batch', 'streaming'), Folds=5
Streaming batch-size sweep=(512, 2048, 8192)
Memory profiling enabled=True


## Helpers

- `fit` mode trains once on the whole training fold.
- `partial_fit` mode streams mini-batches through the same fold (first chunk passes `classes`, subsequent chunks omit it).
- Streaming is evaluated across a sweep of `STREAM_BATCH_SIZES`.
- `train_time_s` is measured without `tracemalloc`; `train_time_mem_s` and `peak_mem_mb` are measured in a separate profiled run.


In [22]:
def build_estimator(solver: str) -> LinearDiscriminantAnalysis:
    return LinearDiscriminantAnalysis(solver=solver, **SOLVER_PARAMS[solver])


def fit_streaming(
    estimator: LinearDiscriminantAnalysis,
    X_train: np.ndarray,
    y_train: np.ndarray,
    classes: np.ndarray,
    batch_size: int,
) -> LinearDiscriminantAnalysis:
    n_samples = X_train.shape[0]
    for start in range(0, n_samples, batch_size):
        stop = min(start + batch_size, n_samples)
        X_chunk = X_train[start:stop]
        y_chunk = y_train[start:stop]
        if start == 0:
            estimator.partial_fit(
                X_chunk,
                y_chunk,
                classes=classes,
            )
        else:
            estimator.partial_fit(
                X_chunk,
                y_chunk,
            )
    return estimator


def benchmark_training(train_builder, *, measure_memory: bool):
    t0 = perf_counter()
    model = train_builder()
    train_time_s = perf_counter() - t0

    train_time_mem_s = np.nan
    peak_mem_mb = np.nan
    if measure_memory:
        tracemalloc.start()
        t1 = perf_counter()
        _ = train_builder()
        train_time_mem_s = perf_counter() - t1
        _, peak_bytes = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        peak_mem_mb = peak_bytes / (1024 ** 2)

    return model, train_time_s, train_time_mem_s, peak_mem_mb


## Run 5-fold holdout benchmark


In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
records = []

for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y), start=1):
    X_train, y_train = X[train_idx], y[train_idx]
    X_test, y_test = X[test_idx], y[test_idx]

    rng = np.random.default_rng(SEED + fold_idx)
    order = rng.permutation(X_train.shape[0])
    X_train_stream = X_train[order]
    y_train_stream = y_train[order]

    for solver in SOLVERS:
        batch_builder = lambda solver=solver: build_estimator(solver).fit(X_train, y_train)
        batch_model, train_time_s, train_time_mem_s, peak_mem_mb = benchmark_training(
            batch_builder,
            measure_memory=MEASURE_MEMORY,
        )
        batch_acc = accuracy_score(y_test, batch_model.predict(X_test))
        records.append(
            {
                'fold': fold_idx,
                'solver': solver,
                'mode': 'batch',
                'batch_size': BATCH_FIT_LABEL,
                'accuracy': batch_acc,
                'train_time_s': train_time_s,
                'train_time_mem_s': train_time_mem_s,
                'peak_mem_mb': peak_mem_mb,
            }
        )

        for batch_size in STREAM_BATCH_SIZES:
            streaming_builder = lambda solver=solver, batch_size=batch_size: fit_streaming(
                build_estimator(solver),
                X_train_stream,
                y_train_stream,
                classes=CLASSES,
                batch_size=batch_size,
            )
            stream_model, train_time_s, train_time_mem_s, peak_mem_mb = benchmark_training(
                streaming_builder,
                measure_memory=MEASURE_MEMORY,
            )
            stream_acc = accuracy_score(y_test, stream_model.predict(X_test))
            records.append(
                {
                    'fold': fold_idx,
                    'solver': solver,
                    'mode': 'streaming',
                    'batch_size': int(batch_size),
                    'accuracy': stream_acc,
                    'train_time_s': train_time_s,
                    'train_time_mem_s': train_time_mem_s,
                    'peak_mem_mb': peak_mem_mb,
                }
            )

expected_rows = N_SPLITS * len(SOLVERS) * (1 + len(STREAM_BATCH_SIZES))
print(f'Collected {len(records)} rows (expected {expected_rows}).')
records[:6]


## Aggregate metrics (mean +/- std across folds, by solver/mode/batch size)


In [ ]:
METRICS = ('accuracy', 'train_time_s', 'train_time_mem_s', 'peak_mem_mb')


def summarize_records(records):
    summary = []
    for solver in SOLVERS:
        for mode in MODES:
            batch_sizes = (BATCH_FIT_LABEL,) if mode == 'batch' else STREAM_BATCH_SIZES
            for batch_size in batch_sizes:
                subset = [
                    r for r in records
                    if r['solver'] == solver
                    and r['mode'] == mode
                    and r['batch_size'] == batch_size
                ]
                if not subset:
                    continue
                row = {'solver': solver, 'mode': mode, 'batch_size': batch_size, 'n': len(subset)}
                for metric in METRICS:
                    values = np.array([r[metric] for r in subset], dtype=float)
                    finite = values[np.isfinite(values)]
                    if finite.size == 0:
                        row[f'{metric}_mean'] = np.nan
                        row[f'{metric}_std'] = np.nan
                    elif finite.size == 1:
                        row[f'{metric}_mean'] = finite[0]
                        row[f'{metric}_std'] = 0.0
                    else:
                        row[f'{metric}_mean'] = finite.mean()
                        row[f'{metric}_std'] = finite.std(ddof=1)
                summary.append(row)
    return summary


summary_rows = summarize_records(records)
summary_rows


In [ ]:
def fmt(mean, std, precision=4):
    if not np.isfinite(mean):
        return 'n/a'
    return f"{mean:.{precision}f} +/- {std:.{precision}f}"


def print_summary_table(summary_rows):
    header = (
        f"{'solver':<8} {'mode':<10} {'batch_size':<11} {'accuracy':>18} {'time_s':>18} {'time_mem_s':>18} {'peak_mem_mb':>18}"
    )
    print(header)
    print('-' * len(header))
    for row in summary_rows:
        acc = fmt(row['accuracy_mean'], row['accuracy_std'])
        t = fmt(row['train_time_s_mean'], row['train_time_s_std'])
        tm = fmt(row['train_time_mem_s_mean'], row['train_time_mem_s_std'])
        m = fmt(row['peak_mem_mb_mean'], row['peak_mem_mb_std'], precision=2)
        print(f"{row['solver']:<8} {row['mode']:<10} {str(row['batch_size']):<11} {acc:>18} {t:>18} {tm:>18} {m:>18}")


print_summary_table(summary_rows)


## Visual comparison (streaming batch-size sweep per solver)


In [ ]:
summary_lookup = {(r['solver'], r['mode'], r['batch_size']): r for r in summary_rows}
STREAM_BATCH_SIZES_SORTED = tuple(sorted(int(x) for x in STREAM_BATCH_SIZES))


def plot_streaming_sweep(metric: str, ylabel: str):
    fig, axes = plt.subplots(1, len(SOLVERS), figsize=(5.4 * len(SOLVERS), 4.0), sharey=True)
    if len(SOLVERS) == 1:
        axes = [axes]

    x = np.arange(len(STREAM_BATCH_SIZES_SORTED), dtype=float)
    x_labels = [str(bs) for bs in STREAM_BATCH_SIZES_SORTED]

    for ax, solver in zip(axes, SOLVERS):
        batch_row = summary_lookup[(solver, 'batch', BATCH_FIT_LABEL)]
        stream_rows = [summary_lookup[(solver, 'streaming', bs)] for bs in STREAM_BATCH_SIZES_SORTED]
        means = np.array([row[f'{metric}_mean'] for row in stream_rows], dtype=float)
        stds = np.array([row[f'{metric}_std'] for row in stream_rows], dtype=float)

        ax.errorbar(x, means, yerr=stds, marker='o', linewidth=1.8, capsize=4, label='streaming')

        baseline = batch_row[f'{metric}_mean']
        baseline_std = batch_row[f'{metric}_std']
        if np.isfinite(baseline):
            ax.axhline(baseline, color='tab:gray', linestyle='--', linewidth=1.6, label='batch baseline')
            if np.isfinite(baseline_std) and baseline_std > 0:
                ax.fill_between(
                    [-0.5, len(STREAM_BATCH_SIZES_SORTED) - 0.5],
                    baseline - baseline_std,
                    baseline + baseline_std,
                    color='tab:gray',
                    alpha=0.12,
                )

        ax.set_title(f"Solver: {solver}")
        ax.set_xlabel('Streaming batch size')
        ax.set_xticks(x)
        ax.set_xticklabels(x_labels, rotation=20)
        ax.grid(alpha=0.25, axis='y')

    axes[0].set_ylabel(ylabel)
    axes[0].legend(loc='best')
    fig.suptitle(f'LDA {metric}: streaming sweep with batch baseline', y=1.02)
    plt.tight_layout()
    plt.show()


plot_streaming_sweep('train_time_s', 'Training time (seconds, no tracemalloc)')
if MEASURE_MEMORY:
    plot_streaming_sweep('train_time_mem_s', 'Training time (seconds, with tracemalloc)')
    plot_streaming_sweep('peak_mem_mb', 'Peak Python memory (MB)')
plot_streaming_sweep('accuracy', 'Holdout accuracy')


## Quick takeaways from this run


In [ ]:
def best(summary_rows, metric: str, higher_is_better: bool):
    key = f'{metric}_mean'
    valid = [row for row in summary_rows if np.isfinite(row[key])]
    if not valid:
        return None
    fn = max if higher_is_better else min
    return fn(valid, key=lambda row: row[key])


def describe_row(row):
    if row['mode'] == 'batch':
        return f"{row['solver']} batch ({row['batch_size']})"
    return f"{row['solver']} streaming (batch={row['batch_size']})"


best_acc = best(summary_rows, 'accuracy', higher_is_better=True)
fastest = best(summary_rows, 'train_time_s', higher_is_better=False)
lowest_mem = best(summary_rows, 'peak_mem_mb', higher_is_better=False)

if best_acc is not None:
    print('Best accuracy:', describe_row(best_acc), f"({best_acc['accuracy_mean']:.4f})")
if fastest is not None:
    print('Fastest training (no tracemalloc):', describe_row(fastest), f"({fastest['train_time_s_mean']:.4f} s)")
if lowest_mem is not None:
    print('Lowest peak memory:', describe_row(lowest_mem), f"({lowest_mem['peak_mem_mb_mean']:.2f} MB)")


## Next steps

- Set `MAX_SAMPLES = None` to run the full 70,000-sample benchmark once runtime is acceptable.
- Expand `STREAM_BATCH_SIZES` (or shift toward larger values) to further map throughput/memory regimes.
- Add another OpenML dataset (for example `Fashion-MNIST`) to test whether solver rankings are dataset-specific.
